# Kinematics of Differential Drive Robots and Wheel Odometry

This notebook covers two complementary models used to predict the pose of a differential drive robot:

1. **Velocity-based (dead reckoning)** &mdash; integrate the linear and angular velocity of the body.
2. **Odometry-based** &mdash; given two consecutive pose estimates, decompose the motion into a first rotation, a translation, and a second rotation.

> Several of the formulas in the original markdown contained typos or sign errors. Each fixed equation is annotated with **Correction** notes below.

## 1. Velocity-based (dead reckoning)

Let $V$ be the linear velocity of the robot's center, $\omega$ the angular velocity about the Instantaneous Center of Curvature ($\mathrm{ICC}$), $R$ the signed turning radius (distance from $\mathrm{ICC}$ to the robot center), and $L$ the wheel base (distance between the two wheels).

The linear velocity of a point traveling on a circle of radius $R$ with angular velocity $\omega$ is

$$V = R\,\omega.$$

Both wheels share the same $\omega$ (rigid body) but trace circles of different radii. With $\mathrm{ICC}$ on the **left** of the robot, the right wheel is at distance $R + L/2$ and the left wheel at distance $R - L/2$:

$$V_r = \omega\!\left(R + \tfrac{L}{2}\right),$$

$$V_l = \omega\!\left(R - \tfrac{L}{2}\right).$$

Adding the two equations gives the body linear velocity and the angular velocity:

$$V_r + V_l = 2\,\omega R \quad\Longrightarrow\quad V = \frac{V_r + V_l}{2}, \qquad \omega = \frac{V_r + V_l}{2R}.$$

Subtracting them yields a direct expression for $\omega$ that does not depend on $R$:

$$V_r - V_l = \omega L \quad\Longrightarrow\quad \boxed{\;\omega = \frac{V_r - V_l}{L}\;}.$$

> **Correction.** The original document had $\omega = (V_l - V_r)/L$, which has the wrong sign.

Dividing the sum by the difference gives the turning radius:

$$\boxed{\;R = \frac{L}{2}\,\frac{V_r + V_l}{V_r - V_l}\;}.$$

> **Correction.** The original document wrote this as $\omega = L\,\frac{V_l + V_r}{2(V_r - V_l)}$. The left-hand side is $R$, not $\omega$ (and the numerator should be $V_r + V_l$, which is the same thing written in a different order).

In [ ]:
import numpy as np

# Sanity check: pick (R, omega, L), compute (V_l, V_r), invert and confirm.
R_true     = 1.5   # m  (ICC to robot center)
omega_true = 0.4   # rad/s
L          = 0.5   # m  (wheel base)

V_r = omega_true * (R_true + L/2)
V_l = omega_true * (R_true - L/2)

V_recovered     = (V_r + V_l) / 2
omega_recovered = (V_r - V_l) / L
R_recovered     = (L/2) * (V_r + V_l) / (V_r - V_l)

print(f"V (linear)  = {V_recovered:.6f}   expected {R_true*omega_true:.6f}")
print(f"omega       = {omega_recovered:.6f}   expected {omega_true:.6f}")
print(f"R           = {R_recovered:.6f}   expected {R_true:.6f}")

assert np.isclose(V_recovered,     R_true*omega_true)
assert np.isclose(omega_recovered, omega_true)
assert np.isclose(R_recovered,     R_true)

### Coordinates of the ICC

If the robot is at $(x, y)$ with heading $\phi$ (CCW from the world $x$-axis), then $\mathrm{ICC}$ lies a distance $R$ to the left of the robot, i.e. at angle $\phi + \pi/2$ from the robot's position:

$$\mathrm{ICC} = \begin{cases}
\mathrm{ICC}_x = x + R\cos\!\left(\phi + \tfrac{\pi}{2}\right) = x - R\sin\phi,\\[4pt]
\mathrm{ICC}_y = y + R\sin\!\left(\phi + \tfrac{\pi}{2}\right) = y + R\cos\phi.
\end{cases}$$

> **Correction.** The original document wrote the intermediate step as $y - R\sin(\pi/2 - \phi)$, which simplifies to $y - R\cos\phi$ &mdash; the opposite sign from the final answer $y + R\cos\phi$. Using the angle $\phi + \pi/2$ (not $\pi/2 - \phi$) makes the derivation consistent.

![Differential drive geometry](images/Differential_Drive_Kinematics_of_a_Wheeled_Mobile_Robot.svg)

### 1.1 Forward Kinematics

Express the pose in the $\mathrm{ICC}$ frame, rotate by $\omega\,\delta t$ about $\mathrm{ICC}$, then convert back to world coordinates:

$$\begin{bmatrix} x' \\ y' \end{bmatrix} =
\begin{bmatrix} \cos(\omega\,\delta t) & -\sin(\omega\,\delta t) \\ \sin(\omega\,\delta t) & \cos(\omega\,\delta t) \end{bmatrix}
\begin{bmatrix} x - \mathrm{ICC}_x \\ y - \mathrm{ICC}_y \end{bmatrix}
+ \begin{bmatrix} \mathrm{ICC}_x \\ \mathrm{ICC}_y \end{bmatrix}.$$

Adding orientation gives the full SE(2) update:

$$\begin{bmatrix} x' \\ y' \\ \phi' \end{bmatrix} =
\begin{bmatrix} \cos(\omega\,\delta t) & -\sin(\omega\,\delta t) & 0 \\ \sin(\omega\,\delta t) & \cos(\omega\,\delta t) & 0 \\ 0 & 0 & 1 \end{bmatrix}
\begin{bmatrix} x - \mathrm{ICC}_x \\ y - \mathrm{ICC}_y \\ \phi \end{bmatrix} +
\begin{bmatrix} \mathrm{ICC}_x \\ \mathrm{ICC}_y \\ \omega\,\delta t \end{bmatrix}.$$

> **Corrections.** The original document had two typos in this matrix: `cos(ωδt θ` (extra $\theta$, missing parenthesis) in entry $(2,2)$, and the bottom row written as `0 1 1` instead of `0 0 1`.

Substitute $x - \mathrm{ICC}_x = R\sin\phi$ and $y - \mathrm{ICC}_y = -R\cos\phi$:

$$\begin{aligned}
x' &= \cos(\omega\delta t)\,(R\sin\phi) - \sin(\omega\delta t)\,(-R\cos\phi) + \mathrm{ICC}_x \\
   &= R\bigl[\sin\phi\cos(\omega\delta t) + \cos\phi\sin(\omega\delta t)\bigr] + \mathrm{ICC}_x \\
   &= R\sin(\phi + \omega\delta t) + \mathrm{ICC}_x,\\[6pt]
y' &= \sin(\omega\delta t)\,(R\sin\phi) + \cos(\omega\delta t)\,(-R\cos\phi) + \mathrm{ICC}_y \\
   &= -R\bigl[\cos\phi\cos(\omega\delta t) - \sin\phi\sin(\omega\delta t)\bigr] + \mathrm{ICC}_y \\
   &= -R\cos(\phi + \omega\delta t) + \mathrm{ICC}_y.
\end{aligned}$$

> **Correction.** The original intermediate expression for $y'$ was written as $-R(\sin(\omega\delta t)\sin\phi + \cos(\omega\delta t)\cos\phi)$, which equals $-R\cos(\omega\delta t - \phi)$. The correct expansion has a minus sign between the two products, $-R(\cos(\omega\delta t)\cos\phi - \sin(\omega\delta t)\sin\phi) = -R\cos(\phi + \omega\delta t)$.

Substituting $\mathrm{ICC}_x = x - R\sin\phi$ and $\mathrm{ICC}_y = y + R\cos\phi$ collapses the result to the canonical form:

$$\boxed{\;\begin{bmatrix} x' \\ y' \\ \phi' \end{bmatrix} =
\begin{bmatrix} x \\ y \\ \phi \end{bmatrix} +
\begin{bmatrix} R\sin(\phi + \omega\delta t) - R\sin\phi \\ -R\cos(\phi + \omega\delta t) + R\cos\phi \\ \omega\,\delta t \end{bmatrix}.\;}$$

Using $R = V/\omega$ (valid only when $\omega \neq 0$):

$$\begin{bmatrix} x' \\ y' \\ \phi' \end{bmatrix} =
\begin{bmatrix} x \\ y \\ \phi \end{bmatrix} +
\begin{bmatrix} \tfrac{V}{\omega}\bigl[\sin(\phi + \omega\delta t) - \sin\phi\bigr] \\ -\tfrac{V}{\omega}\bigl[\cos(\phi + \omega\delta t) - \cos\phi\bigr] \\ \omega\,\delta t \end{bmatrix}.$$

When $\omega \to 0$, both expressions tend to the straight-line update

$$x' = x + V\,\delta t\,\cos\phi,\qquad y' = y + V\,\delta t\,\sin\phi,\qquad \phi' = \phi.$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def step(pose, V, omega, dt):
    x, y, phi = pose
    if abs(omega) < 1e-9:
        return np.array([x + V*dt*np.cos(phi), y + V*dt*np.sin(phi), phi])
    R = V / omega
    return np.array([
        x + R*(np.sin(phi + omega*dt) - np.sin(phi)),
        y - R*(np.cos(phi + omega*dt) - np.cos(phi)),
        phi + omega*dt,
    ])

# Sanity check 1: V=1, omega=0.5 should trace a circle of radius R = V/omega = 2.
pose = np.array([0.0, 0.0, 0.0])
traj = [pose]
dt = 0.05
for _ in range(int(2*np.pi / 0.5 / dt)):
    pose = step(pose, 1.0, 0.5, dt)
    traj.append(pose)
traj = np.array(traj)

# Center of the circle should be at (0, 2) (ICC to the left of the start pose).
radii = np.linalg.norm(traj[:, :2] - np.array([0.0, 2.0]), axis=1)
print(f"Trajectory radius: mean={radii.mean():.6f}, std={radii.std():.2e} (expected 2.0)")
assert np.allclose(radii, 2.0, atol=1e-9)

plt.figure(figsize=(4, 4))
plt.plot(traj[:, 0], traj[:, 1])
plt.scatter([0.0], [2.0], c='r', label='ICC')
plt.axis('equal'); plt.grid(True); plt.legend()
plt.title('Forward kinematics: V=1, omega=0.5')
plt.show()

### 1.2 Inverse Kinematics of Differential Drive Robots

Given a desired body twist $(V, \omega)$, recover the wheel speeds:

$$V_r = V + \tfrac{L}{2}\omega, \qquad V_l = V - \tfrac{L}{2}\omega.$$

These follow directly from the forward equations $V_{r,l} = \omega(R \pm L/2)$ together with $V = R\omega$.

Reference: [icckinematics.pdf (Columbia)](https://www.cs.columbia.edu/~allen/F17/NOTES/icckinematics.pdf).

## 2. Odometry-based Motion Model

![Odometry model](images/odometry_model.jpg)

Two consecutive odometry readings $(\bar x, \bar y, \bar\theta)$ and $(\bar x', \bar y', \bar\theta')$ are decomposed into **rotation, translation, rotation**:

$$\delta_{\text{rot1}} = \operatorname{atan2}\bigl(\bar y' - \bar y,\; \bar x' - \bar x\bigr) - \bar\theta,$$

$$\delta_{\text{trans}} = \sqrt{(\bar x' - \bar x)^2 + (\bar y' - \bar y)^2},$$

$$\delta_{\text{rot2}} = \bar\theta' - \bar\theta - \delta_{\text{rot1}}.$$

> Compared to the original markdown: a stray closing parenthesis in the $\delta_{\text{rot1}}$ formula has been removed.

### Motion model

Apply the same three-step motion to the previous belief mean $(x_{t-1}, y_{t-1}, \theta_{t-1})$:

$$g(u_t, \mu_{t-1}) = \begin{pmatrix} x_t \\ y_t \\ \theta_t \end{pmatrix} =
\begin{pmatrix} x_{t-1} \\ y_{t-1} \\ \theta_{t-1} \end{pmatrix} +
\begin{pmatrix} \delta_{\text{trans}}\,\cos(\theta_{t-1} + \delta_{\text{rot1}}) \\ \delta_{\text{trans}}\,\sin(\theta_{t-1} + \delta_{\text{rot1}}) \\ \delta_{\text{rot1}} + \delta_{\text{rot2}} \end{pmatrix}.$$

### Jacobian $G_t$ (with respect to the state)

$$G_t = \frac{\partial g}{\partial (x, y, \theta)^{\!\top}}.$$

Only the $\theta$-column has non-trivial cross terms (the translation direction depends on the previous heading):

$$G_t = I + \begin{pmatrix}
0 & 0 & -\delta_{\text{trans}}\,\sin(\theta + \delta_{\text{rot1}}) \\
0 & 0 & \phantom{-}\delta_{\text{trans}}\,\cos(\theta + \delta_{\text{rot1}}) \\
0 & 0 & 0
\end{pmatrix} =
\begin{pmatrix}
1 & 0 & -\delta_{\text{trans}}\,\sin(\theta + \delta_{\text{rot1}}) \\
0 & 1 & \phantom{-}\delta_{\text{trans}}\,\cos(\theta + \delta_{\text{rot1}}) \\
0 & 0 & 1
\end{pmatrix}.$$

The original markdown said "not sure about the followings" &mdash; the Jacobian above is correct, as verified numerically below.

In [ ]:
import numpy as np

def g(state, u):
    x, y, th = state
    d_rot1, d_trans, d_rot2 = u
    return np.array([
        x + d_trans * np.cos(th + d_rot1),
        y + d_trans * np.sin(th + d_rot1),
        th + d_rot1 + d_rot2,
    ])

def G_analytical(state, u):
    _, _, th = state
    d_rot1, d_trans, _ = u
    return np.array([
        [1.0, 0.0, -d_trans * np.sin(th + d_rot1)],
        [0.0, 1.0,  d_trans * np.cos(th + d_rot1)],
        [0.0, 0.0,  1.0],
    ])

def G_numerical(state, u, eps=1e-6):
    J = np.zeros((3, 3))
    for i in range(3):
        e = np.zeros(3); e[i] = eps
        J[:, i] = (g(state + e, u) - g(state - e, u)) / (2 * eps)
    return J

rng = np.random.default_rng(0)
for _ in range(5):
    state = rng.standard_normal(3)
    u     = rng.standard_normal(3)
    Ga = G_analytical(state, u)
    Gn = G_numerical(state, u)
    err = np.max(np.abs(Ga - Gn))
    print(f"max |G_analytical - G_numerical| = {err:.2e}")
    assert err < 1e-7

### Range-bearing observation model

For a landmark $j$ with map position $(\bar\mu_{j,x}, \bar\mu_{j,y})$ and a robot pose $\bar\mu_t = (\bar\mu_{t,x}, \bar\mu_{t,y}, \bar\mu_{t,\theta})$, the predicted measurement is

$$h(\bar\mu_t, j) = \begin{pmatrix} r_t^i \\ \phi_t^i \end{pmatrix} =
\begin{pmatrix}
\sqrt{(\bar\mu_{j,x} - \bar\mu_{t,x})^2 + (\bar\mu_{j,y} - \bar\mu_{t,y})^2} \\
\operatorname{atan2}\bigl(\bar\mu_{j,y} - \bar\mu_{t,y},\; \bar\mu_{j,x} - \bar\mu_{t,x}\bigr) - \bar\mu_{t,\theta}
\end{pmatrix},$$

where the components are the **range** and the **bearing relative to the robot heading**, respectively. The observed pair is

$$z_t^i = \begin{bmatrix} r_t^i \\ \phi_t^i \end{bmatrix}.$$

> Compared to the original markdown: a stray closing parenthesis in the $\operatorname{atan2}$ expression has been removed.

## DiffBot Differential Drive Mobile Robot

[ros-mobile-robots](https://ros-mobile-robots.com/)

## Summary of corrections vs. the original markdown

| # | Where | Original | Correct |
|---|---|---|---|
| 1 | sec. 1, $\omega$ from wheel-speed difference | $\omega = (V_l - V_r)/L$ | $\omega = (V_r - V_l)/L$ |
| 2 | sec. 1, ratio formula | $\omega = L\,(V_l + V_r) / [2(V_r - V_l)]$ | $R = (L/2)\,(V_r + V_l) / (V_r - V_l)$ |
| 3 | sec. 1, $\mathrm{ICC}_y$ intermediate | $y - R\sin(\pi/2 - \phi)$ (gives $-R\cos\phi$) | $y + R\sin(\pi/2 + \phi) = y + R\cos\phi$ |
| 4 | sec. 1.1, 3D rotation matrix | entry $(2,2)$ written `cos(omega*dt theta`; bottom row `0 1 1` | entry $(2,2) = \cos(\omega\delta t)$; bottom row $0\;0\;1$ |
| 5 | sec. 1.1, expansion of $y'$ | $-R(\sin(\omega\delta t)\sin\phi + \cos(\omega\delta t)\cos\phi)$ | $-R(\cos(\omega\delta t)\cos\phi - \sin(\omega\delta t)\sin\phi) = -R\cos(\phi + \omega\delta t)$ |
| 6 | sec. 2, $\delta_{\text{rot1}}$ | extra `)` after the `atan2` arguments | parentheses balanced |
| 7 | sec. 2, observation model | extra `)` after the `atan2` arguments | parentheses balanced |

The Jacobian $G_t$ in the original was correct (the document only flagged it as uncertain); the numerical check above confirms this.